1. Research and compare different configuration management tools (Ansible, Puppet, Chef) and write a report on their use cases and advantages?
```
1. ANSIBLE
Use Cases:
*IT orchestration and configuration management
*Application deployment
*Server provisioning
*Cloud infrastructure automation
*Network automation
*Security compliance and patching

Advantages:
*Agentless Architecture: No agent installation required on target machines, reduces overhead
*Python-Based: Easy to learn and extend with Python knowledge
*YAML Syntax: Simple, human-readable playbooks (YAML format)
*Push-Based Model: Control server pushes configurations directly
*Low Overhead: Minimal resource requirements
*Excellent Documentation: Comprehensive and community-driven
*Rapid Deployment: Quick setup and execution
*Wide Platform Support: Works on Linux, Windows, macOS, cloud platforms

Playbook Example:
---
- name: Install and configure web server
  hosts: webservers
  tasks:
    - name: Install Apache
      apt:
        name: apache2
        state: present
    - name: Start Apache service
      service:
        name: apache2
        state: started
        enabled: yes


2. PUPPET
Use Cases:
*Large-scale infrastructure management
*Complex dependency management
*Enterprise IT operations
*Compliance and governance enforcement
*Multi-environment deployments

Advantages:
*Pull-Based Architecture: Agents pull configurations from master server
*Declarative Model: Strong focus on desired state
*Powerful DSL: Domain Specific Language for complex configurations
*Scalability: Excellent for managing thousands of nodes
*Mature Ecosystem: Established in enterprise environments
*Master-Agent Model: Centralized control with distributed agents
*Strong Reporting: Detailed compliance and configuration reports

Manifest Example:
class apache {
  package { 'apache2':
    ensure => installed,
  }

  service { 'apache2':
    ensure  => running,
    enable  => true,
    require => Package['apache2'],
  }
}


3. CHEF
Use Cases:
*Infrastructure as Code (IaC)
*Complex application deployments
*Cloud provisioning (AWS, Azure, GCP)
*Development and testing automation
*Multi-tier application management

Advantages:
*Chef DSL: Based on Ruby, powerful and flexible
*Recipe-Based Approach: Highly modular and reusable
*Community Cookbooks: Large library of pre-built configurations
*Version Control Integration: Natural fit with Git workflows
*Powerful Templating: Ruby-based for dynamic configurations
*Testing Support: Built-in testing frameworks
*Multi-Cloud Support: Works across major cloud platforms

Recipe Example:
package 'apache2' do
  action :install
end

service 'apache2' do
  action [:enable, :start]
end
```
2. Extend the CI pipeline to include continuous delivery using Jenkins or GitLab CI. Implement a blue-green deployment strategy?
```
Blue-Green deployment is a release technique where two identical production environments are maintained:
*Blue Environment: Current live production
*Green Environment: New version staging
*Switch: Traffic redirects to green once validated

Jenkins Pipeline Configuration:
Jenkinsfile (Declarative Pipeline):
pipeline {
    agent any
    
    parameters {
        choice(name: 'ENVIRONMENT', choices: ['blue', 'green'], description: 'Target Environment')
    }
    
    environment {
        DOCKER_REGISTRY = "docker.io"
        APP_NAME = "myapp"
        VERSION = "${BUILD_NUMBER}"
    }
    
    stages {
        stage('Checkout') {
            steps {
                git branch: 'main', url: 'https://github.com/yourrepo/myapp.git'
            }
        }
        
        stage('Build') {
            steps {
                script {
                    sh '''
                        echo "Building application version ${VERSION}"
                        docker build -t ${DOCKER_REGISTRY}/${APP_NAME}:${VERSION} .
                    '''
                }
            }
        }
        
        stage('Unit Tests') {
            steps {
                sh 'npm test || mvn test'
            }
        }
        
        stage('Build Docker Image') {
            steps {
                sh '''
                    docker build -t ${DOCKER_REGISTRY}/${APP_NAME}:${VERSION} .
                    docker tag ${DOCKER_REGISTRY}/${APP_NAME}:${VERSION} ${DOCKER_REGISTRY}/${APP_NAME}:latest
                '''
            }
        }
        
        stage('Push to Registry') {
            steps {
                sh '''
                    docker login -u ${DOCKER_USERNAME} -p ${DOCKER_PASSWORD}
                    docker push ${DOCKER_REGISTRY}/${APP_NAME}:${VERSION}
                    docker push ${DOCKER_REGISTRY}/${APP_NAME}:latest
                '''
            }
        }
        
        stage('Deploy to Green Environment') {
            when {
                expression { params.ENVIRONMENT == 'green' }
            }
            steps {
                script {
                    sh '''
                        echo "Deploying to Green environment"
                        kubectl set image deployment/myapp-green \
                            myapp=${DOCKER_REGISTRY}/${APP_NAME}:${VERSION} \
                            -n production
                        kubectl rollout status deployment/myapp-green -n production
                    '''
                }
            }
        }
        
        stage('Smoke Tests on Green') {
            steps {
                script {
                    sh '''
                        echo "Running smoke tests on Green environment"
                        curl -f http://green-app.example.com/health || exit 1
                        ./run_smoke_tests.sh green
                    '''
                }
            }
        }
        
        stage('Performance Tests') {
            steps {
                sh '''
                    echo "Running performance and load tests"
                    ./run_load_tests.sh green
                '''
            }
        }
        
        stage('Manual Approval') {
            steps {
                input 'Approve deployment to production?'
            }
        }
        
        stage('Switch Traffic (Blue to Green)') {
            steps {
                script {
                    sh '''
                        echo "Switching traffic from Blue to Green"
                        kubectl patch service myapp-service -n production \
                            -p '{"spec":{"selector":{"version":"green"}}}'
                        
                        echo "Waiting for connection draining..."
                        sleep 30
                    '''
                }
            }
        }
        
        stage('Verify Live Traffic') {
            steps {
                sh '''
                    echo "Verifying production environment"
                    curl -f http://myapp.example.com/health || exit 1
                    ./run_smoke_tests.sh production
                '''
            }
        }
        
        stage('Rollback if Needed') {
            when {
                expression { currentBuild.result == 'FAILURE' }
            }
            steps {
                script {
                    sh '''
                        echo "Rolling back to Blue environment"
                        kubectl patch service myapp-service -n production \
                            -p '{"spec":{"selector":{"version":"blue"}}}'
                    '''
                }
            }
        }
    }
    
    post {
        always {
            cleanWs()
        }
        success {
            echo "Deployment successful!"
            // Send notification
        }
        failure {
            echo "Deployment failed - automatic rollback triggered"
        }
    }
}

Kubernetes Deployment Configuration:
blue-green-deployment.yaml:
---
# Blue Deployment
apiVersion: apps/v1
kind: Deployment
metadata:
  name: myapp-blue
  namespace: production
spec:
  replicas: 3
  selector:
    matchLabels:
      app: myapp
      version: blue
  template:
    metadata:
      labels:
        app: myapp
        version: blue
    spec:
      containers:
      - name: myapp
        image: docker.io/myapp:latest
        ports:
        - containerPort: 8080
        readinessProbe:
          httpGet:
            path: /health
            port: 8080
          initialDelaySeconds: 10
          periodSeconds: 5
---
# Green Deployment
apiVersion: apps/v1
kind: Deployment
metadata:
  name: myapp-green
  namespace: production
spec:
  replicas: 3
  selector:
    matchLabels:
      app: myapp
      version: green
  template:
    metadata:
      labels:
        app: myapp
        version: green
    spec:
      containers:
      - name: myapp
        image: docker.io/myapp:latest
        ports:
        - containerPort: 8080
        readinessProbe:
          httpGet:
            path: /health
            port: 8080
          initialDelaySeconds: 10
          periodSeconds: 5
---
# Service (Initially pointing to Blue)
apiVersion: v1
kind: Service
metadata:
  name: myapp-service
  namespace: production
spec:
  selector:
    app: myapp
    version: blue  # Switch this to 'green' after verification
  ports:
  - protocol: TCP
    port: 80
    targetPort: 8080
  type: LoadBalancer


Jenkins Configuration Scripts:
health-check.sh:
#!/bin/bash
ENVIRONMENT=$1
MAX_RETRIES=30
RETRY_INTERVAL=2

for i in $(seq 1 $MAX_RETRIES); do
    if curl -s http://${ENVIRONMENT}-app.example.com/health | grep -q "healthy"; then
        echo "✓ ${ENVIRONMENT} environment is healthy"
        exit 0
    fi
    echo "Attempt $i: Waiting for ${ENVIRONMENT} to be ready..."
    sleep $RETRY_INTERVAL
done

echo "✗ ${ENVIRONMENT} environment health check failed"
exit 1

rollback.sh:
#!/bin/bash
echo "Initiating rollback to Blue environment..."
kubectl patch service myapp-service -n production \
    -p '{"spec":{"selector":{"version":"blue"}}}'
echo "Traffic redirected to Blue environment"
```